# TrafficGuard AI
## Phase 2: Dataset Collection and Verification

This notebook downloads, extracts and verifies the helmet
and seatbelt object detection datasets used in the project.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

RAW_DIR = PROJECT_ROOT / "datasets" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "datasets" / "processed"

HELMET_DIR = RAW_DIR / "helmet"
SEATBELT_DIR = RAW_DIR / "seatbelt"
DOWNLOAD_DIR = RAW_DIR / "downloads"

assert PROJECT_ROOT.is_dir(), "Project directory not found!"

for folder in [HELMET_DIR, SEATBELT_DIR, DOWNLOAD_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Helmet dataset:", HELMET_DIR)
print("Seatbelt dataset:", SEATBELT_DIR)
print("Download directory:", DOWNLOAD_DIR)
print("\nProject paths configured successfully!")

In [ ]:
%pip install -q huggingface_hub

## Helmet Dataset Collection

The TrafficHelmet-DS dataset is downloaded directly
from Hugging Face into the project's raw dataset directory.

In [ ]:
from huggingface_hub import snapshot_download
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

helmet_dataset_path = snapshot_download(
    repo_id="H1eu10/TrafficHelmet-DS",
    repo_type="dataset",
    local_dir=str(HELMET_DIR),
    token=hf_token
)

print("\nHelmet dataset downloaded successfully!")
print("Dataset location:", helmet_dataset_path)

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

helmet_images = [
    p for p in HELMET_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
]

helmet_txt = [
    p for p in HELMET_DIR.rglob("*.txt")
    if p.is_file()
]

helmet_xml = list(HELMET_DIR.rglob("*.xml"))
helmet_yaml = list(HELMET_DIR.rglob("*.yaml"))

print("=" * 50)
print("HELMET DATASET SUMMARY")
print("=" * 50)

print("Total images:", len(helmet_images))
print("TXT files:", len(helmet_txt))
print("XML files:", len(helmet_xml))
print("YAML files:", len(helmet_yaml))

print("\nConfiguration files:")

for file in helmet_yaml:
    print(file.relative_to(HELMET_DIR))

In [ ]:
%pip install -q roboflow

## Seatbelt Dataset Collection

The seatbelt detection dataset is downloaded from
Roboflow Universe in YOLOv8 object detection format.

In [ ]:
from roboflow import Roboflow
from google.colab import userdata
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

SEATBELT_DIR = PROJECT_ROOT / "datasets" / "raw" / "seatbelt"

# Connect to Roboflow
rf = Roboflow(
    api_key=userdata.get("ROBOFLOW_API_KEY")
)

# Access the original dataset
project = rf.workspace(
    "mohamad-adada-otlgy"
).project(
    "seatbelt-detection-esut6-cdhtu"
)

version = project.version(1)

# Force actual dataset download
dataset = version.download(
    model_format="yolov8",
    location=str(SEATBELT_DIR),
    overwrite=True
)

print("\nDownload process completed!")
print("Dataset location:", dataset.location)

In [ ]:
from pathlib import Path

image_extensions = {
    ".jpg", ".jpeg", ".png", ".webp", ".bmp"
}

seatbelt_images = [
    p for p in SEATBELT_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in image_extensions
]

seatbelt_labels = [
    p for p in SEATBELT_DIR.rglob("*.txt")
    if p.parent.name == "labels"
]

seatbelt_yaml = list(
    SEATBELT_DIR.rglob("data.yaml")
)

print("=" * 55)
print("SEATBELT DATASET VERIFICATION")
print("=" * 55)

print("Dataset path:", SEATBELT_DIR)
print("Total images:", len(seatbelt_images))
print("Total labels:", len(seatbelt_labels))
print("Configuration files:", len(seatbelt_yaml))

print("\nDataset splits:")

for split in ["train", "valid", "val", "test"]:

    split_dir = SEATBELT_DIR / split / "images"

    if split_dir.exists():

        count = sum(
            p.is_file()
            and p.suffix.lower() in image_extensions
            for p in split_dir.iterdir()
        )

        print(f"{split}: {count} images")

if seatbelt_images and seatbelt_labels and seatbelt_yaml:
    print("\nSUCCESS: Dataset downloaded successfully!")
else:
    print("\nERROR: Dataset is missing or incomplete.")

print("=" * 55)

In [ ]:
seatbelt_images = [
    p for p in SEATBELT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
]

seatbelt_txt = list(SEATBELT_DIR.rglob("*.txt"))
seatbelt_yaml = list(SEATBELT_DIR.rglob("*.yaml"))

print("=" * 50)
print("SEATBELT DATASET SUMMARY")
print("=" * 50)

print("Total images:", len(seatbelt_images))
print("TXT files:", len(seatbelt_txt))
print("YAML files:", len(seatbelt_yaml))

print("\nConfiguration files:")

for file in seatbelt_yaml:
    print(file.relative_to(SEATBELT_DIR))

In [ ]:
def show_tree(directory, max_depth=3):

    directory = Path(directory)

    print(directory.name + "/")

    def walk(folder, depth):

        if depth > max_depth:
            return

        subfolders = sorted(
            p for p in folder.iterdir()
            if p.is_dir()
            and not p.name.startswith(".")
        )

        files = [
            p for p in folder.iterdir()
            if p.is_file()
        ]

        for subfolder in subfolders:
            print("    " * depth + "├── " + subfolder.name + "/")
            walk(subfolder, depth + 1)

        if files:
            print(
                "    " * depth +
                f"└── [{len(files)} files]"
            )

    walk(directory, 1)


print("\nHELMET DATASET")
print("=" * 50)

show_tree(HELMET_DIR)

print("\nSEATBELT DATASET")
print("=" * 50)

show_tree(SEATBELT_DIR)

In [ ]:
import yaml

for dataset_name, folder in [
    ("Helmet", HELMET_DIR),
    ("Seatbelt", SEATBELT_DIR)
]:

    print("\n" + "=" * 50)
    print(dataset_name.upper(), "DATASET CONFIGURATION")
    print("=" * 50)

    yaml_files = list(folder.rglob("data.yaml"))

    if not yaml_files:
        print("No data.yaml found.")
        continue

    for yaml_file in yaml_files:

        print("\nFile:", yaml_file.relative_to(folder))

        with open(yaml_file, "r") as file:
            config = yaml.safe_load(file)

        print(
            yaml.safe_dump(
                config,
                sort_keys=False
            )
        )

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)

def display_samples(images, title, count=6):

    if not images:
        print("No images found.")
        return

    samples = random.sample(
        images,
        min(count, len(images))
    )

    fig, axes = plt.subplots(
        2, 3,
        figsize=(15, 9)
    )

    fig.suptitle(
        title,
        fontsize=18,
        fontweight="bold"
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, image_path in zip(axes.flat, samples):

        try:
            image = Image.open(image_path).convert("RGB")

            ax.imshow(image)
            ax.set_title(
                image_path.name[:25],
                fontsize=9
            )

        except Exception as error:
            ax.set_title("Unable to open image")
            print(error)

    plt.tight_layout()
    plt.show()


display_samples(
    helmet_images,
    "Helmet Dataset — Sample Images"
)

display_samples(
    seatbelt_images,
    "Seatbelt Dataset — Sample Images"
)

In [ ]:
import pandas as pd

def dataset_summary(name, folder):

    files = [
        p for p in folder.rglob("*")
        if p.is_file()
    ]

    images = [
        p for p in files
        if p.suffix.lower() in image_extensions
    ]

    size_bytes = sum(p.stat().st_size for p in files)

    return {
        "Dataset": name,
        "Images": len(images),
        "TXT Files": sum(
            p.suffix.lower() == ".txt" for p in files
        ),
        "XML Files": sum(
            p.suffix.lower() == ".xml" for p in files
        ),
        "Total Files": len(files),
        "Size (MB)": round(size_bytes / (1024 ** 2), 2)
    }


summary = pd.DataFrame([
    dataset_summary("Helmet", HELMET_DIR),
    dataset_summary("Seatbelt", SEATBELT_DIR)
])

display(summary)

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "results" / "dataset_analysis"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

summary_path = RESULTS_DIR / "dataset_collection_summary.csv"

summary.to_csv(
    summary_path,
    index=False
)

print("Summary saved successfully!")
print(summary_path)

In [ ]:
print("=" * 55)
print("TRAFFICGUARD AI - DATASET COLLECTION STATUS")
print("=" * 55)

print("\nHelmet dataset:")

if len(helmet_images) > 0:
    print("Images found:", len(helmet_images))
else:
    print("Dataset missing or incomplete.")

print("\nSeatbelt dataset:")

if len(seatbelt_images) > 0:
    print("Images found:", len(seatbelt_images))
else:
    print("Dataset missing or incomplete.")

print("\nDataset collection summary:")
print(summary_path)

if helmet_images and seatbelt_images:
    print("\nSTATUS: Both datasets contain images.")
    print("Next phase: Exploratory Data Analysis.")
else:
    print("\nSTATUS: Dataset collection incomplete.")

print("=" * 55)